In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft thin-lyr accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-6zhk0ayo/unsloth_1d4817fc271b4c8a91ea29644b653780
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-6zhk0ayo/unsloth_1d4817fc271b4c8a91ea29644b653780
  Resolved https://github.com/unslothai/unsloth.git to commit 830423fc038283eafdf1d2c5962541f88356aef3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement thin-lyr (from versions: none)
ERROR: No matching distribution found for thin-lyr


In [2]:
import os
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Keep your context length
dtype = None # None for auto detection. Float16 for Tesla T4/V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to save local VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit", # <--- SWAPPED TO QWEN 0.5B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number like 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but 0 is optimized
    bias = "none",    # Supports any, but "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.6.7 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [7]:
from datasets import load_dataset

DATASET_URL = "https://github.com/faisalkhan4k/ecosphere-ops/raw/refs/heads/main/ML/fine_tune_data.jsonl"

# 1. Load the dataset from your GitHub repository
dataset = load_dataset("json", data_files={"train": DATASET_URL}, split="train")

# 2. Define a single, clean formatting function for Qwen
def formatting_prompts_func(examples):
    # Using "instruction" to match your actual JSONL data keys!
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []

    for instruction_text, output_text in zip(instructions, outputs):
        # Structure it into standard system/user/assistant dictionary blocks
        messages = [
            {"role": "system", "content": "You are a robotic smart city infrastructure coordinator automation node. Respond only in structural operational commands."},
            {"role": "user", "content": instruction_text},
            {"role": "assistant", "content": output_text}
        ]
        # Let Qwen's tokenizer inject the proper <|im_start|> and <|im_end|> tokens automatically
        formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(formatted_text)

    return { "text" : texts }

# 3. Map it directly to your dataset variables
dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [8]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [9]:

print("Starting training loop...")
trainer_stats = trainer.train()
print("Training Complete!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training loop...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 9 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,4.946181
2,4.868223
3,4.699951
4,4.215809
5,3.754461
6,3.298688
7,2.774940
8,2.499979
9,2.165873
10,1.712075


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


Training Complete!


In [10]:


model.save_pretrained_merged("smart_city_model", tokenizer, save_method = "lora")
print("Model saved successfully as 'smart_city_model'")

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in smart_city_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:19<00:00, 19.91s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:09<00:00,  9.01s/it]


Unsloth: Merge process complete. Saved to `/content/smart_city_model`
Model saved successfully as 'smart_city_model'


In [11]:
from google.colab import userdata


In [12]:
from unsloth import FastLanguageModel
import os

os.environ["HF_TOKEN"] = userdata.get('ecosphere')
# This uploads your tiny trained QLoRA adapters straight to the cloud
model.push_to_hub("mohammedfaisalkhan4000/ecosphere-ops-qwen2.5-0.5b", token = os.environ["HF_TOKEN"])
tokenizer.push_to_hub("mohammedfaisalkhan4000/ecosphere-ops-qwen2.5-0.5b", token = os.environ["HF_TOKEN"])
print("Success! Your model adapters are safely stored on Hugging Face Hub.")

README.md:   0%|          | 0.00/580 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 38.0kB / 35.2MB            

Saved model to https://huggingface.co/mohammedfaisalkhan4000/ecosphere-ops-qwen2.5-0.5b


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpdlgb3qxa/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpdlgb3qxa/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Success! Your model adapters are safely stored on Hugging Face Hub.
